In [1]:
""" Imports and Path Setup """
%load_ext autoreload
%autoreload 2

import sys
import os

root = os.path.abspath("..")
if root not in sys.path:
    sys.path.insert(0, root)
    
import pandas as pd
from src.data_loader import load_raw_data, get_data_shape_summary

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
""" Load raw data """

df_raw = load_raw_data()
shape = get_data_shape_summary(df_raw)

print(f"Rows      : {shape['n_rows']:,}")
print(f"Columns   : {shape['n_columns']}")
print(f"Total cells: {shape['n_cells']:,}")

Rows      : 5,001
Columns   : 21
Total cells: 105,021


In [3]:
"""
Display first 5 rows
"""
df_raw.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,CUST-00005,Male,0,Yes,Yes,53,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,29.29,1553.16,No
1,CUST-00001,Male,0,No,Yes,61,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Credit card,87.04,5303.45,No
2,CUST-00002,Female,0,No,No,68,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card,28.63,1944.52,No
3,CUST-00003,Male,0,No,NaN,62,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card,27.85,1736.17,No
4,CUST-00004,Male,1,Yes,No,1,Yes,Yes,DSL,No,Yes,No,No,No,Yes,Month-to-month,Yes,Electronic check,67.92,62.16,Yes


In [4]:
"""
Display last 5 rows
"""
df_raw.tail()

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges,churn
4996,CUST-04996,Male,0,Yes,No,1,Yes,Yes,DSL,No,No,No,No,Yes,No,Two year,No,NaN,64.95,70.65,No
4997,CUST-04997,F,0,Yes,No,66,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,92.65,6109.88,Yes
4998,CUST-04998,Male,1,Yes,Yes,47,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Electronic check,31.21,1465.45,No
4999,CUST-04999,Female,0,No,No,68,Yes,No,Fiber optic,Yes,Yes,Yes,No,No,No,Two year,No,Credit card,97.14,6607.0,No
5000,CUST-05000,Female,0,Yes,Yes,27,Yes,Yes,DSL,Yes,No,No,No,No,Yes,One year,No,Electronic check,72.48,1958.25,No


In [5]:
"""
Copy raw data frame for working.
"""
df = df_raw.copy()
print("Working copy created. df_raw remains untouched.")

Working copy created. df_raw remains untouched.


In [6]:
"""Run profiler"""
from src.profiler import run_full_profile

profile = run_full_profile(df, id_column="customer_id", target_col="churn")
print("Full profile computed.")

Full profile computed.


In [7]:
"""
Save data dictionary
"""
def generate_data_dictionary(df: pd.DataFrame, output_path: str) -> None:
    """
    Generate a data dictionary from the loaded DataFrame and a
    pre-defined description map. Saves as a markdown file to docs/.
    """
    description_map = {
        "customer_id": "Unique identifier per customer. Expected: no duplicates.",
        "gender": "Customer gender. Raw export also contains 'male', 'M' and 'F'; standardised to Male / Female during cleaning.",
        "senior_citizen": "Whether customer is a senior citizen. Values: 0 (No), 1 (Yes). Note: stored as int — will be standardized in cleaning.",
        "partner": "Whether the customer has a partner. Values: Yes / No.",
        "dependents": "Whether the customer has dependents. Values: Yes / No.",
        "tenure_months": "Number of months the customer has been with the company. Range: 0–72.",
        "phone_service": "Whether the customer has a phone service. Values: Yes / No.",
        "multiple_lines": "Whether the customer has multiple phone lines. Values: Yes / No / No phone service.",
        "internet_service": "Type of internet service. Values: DSL / Fiber optic / No.",
        "online_security": "Whether the customer has online security add-on. Values: Yes / No / No internet service.",
        "online_backup": "Whether the customer has online backup add-on. Values: Yes / No / No internet service.",
        "device_protection": "Whether the customer has device protection add-on. Values: Yes / No / No internet service.",
        "tech_support": "Whether the customer has tech support add-on. Values: Yes / No / No internet service.",
        "streaming_tv": "Whether the customer has streaming TV service. Values: Yes / No / No internet service.",
        "streaming_movies": "Whether the customer has streaming movies service. Values: Yes / No / No internet service.",
        "contract_type": "Type of contract. Raw export also contains 'month-to-month'; standardised during cleaning. Values: Month-to-month / One year / Two year.",
        "paperless_billing": "Whether the customer uses paperless billing. Values: Yes / No.",
        "payment_method": "Payment method used. Values: Electronic check / Mailed check / Bank transfer / Credit card.",
        "monthly_charges": "Amount charged to the customer each month. Type: float.",
        "total_charges": "Total amount charged over the customer's tenure. Type: float. Known issue: stored as object in source, with 64 blank values that are all zero-tenure customers rather than missing data.",
        "churn": "TARGET VARIABLE. Whether the customer churned. Values: Yes / No.",
    }

    records = []
    for col in df.columns:
        records.append({
            "Column": col,
            "Stored Type": str(df[col].dtype),
            "Unique Values": df[col].nunique(dropna=True),
            "Missing (NaN)": df[col].isna().sum(),
            "Description": description_map.get(col, "No description provided."),
        })

    dict_df = pd.DataFrame(records)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("# Data Dictionary — Telecom Customer Churn Dataset\n\n")
        f.write(f"**Source:** `data/raw/telecom_customers.csv` \n")
        f.write(f"**Rows:** {len(df):,} | **Columns:** {len(df.columns)} \n\n")
        f.write("---\n\n")
        f.write(dict_df.to_markdown(index=False))
        f.write("\n\n---\n")
        f.write("*Generated programmatically. Do not edit manually.*\n")

    print(f"✅ Data dictionary saved to {output_path}")


generate_data_dictionary(df, output_path="../docs/data_dictionary.md")


✅ Data dictionary saved to ../docs/data_dictionary.md


In [8]:
"""Display profiling summary"""

print("=" * 60)
print("CLASS 1 — PROFILING SUMMARY")
print("=" * 60)

shape = profile["shape"]
dupe = profile["duplicate_report"]
missing = profile["missing_report"]
target = profile["target_distribution"]

print(f"Dataset size         : {shape['n_rows']:,} customers, {shape['n_columns']} attributes")
print(f"Full row duplicates  : {dupe['full_row_duplicates']}")
print(f"ID duplicates        : {dupe['id_duplicates']}")
print(f"Columns with missing : {len(missing[missing['combined_missing'] > 0])}")

churn_yes_pct = target[target['class'] == 'Yes']['percentage'].values
if len(churn_yes_pct) > 0:
    print(f"Overall churn rate   : {churn_yes_pct[0]:.1f}%")

print()
print("📌 Confirmed issues for Class 2 (Cleaning):")
print("  1. total_charges stored as object — needs type conversion")
print("  2. Investigate structured missingness in total_charges")
print("  3. senior_citizen stored as int — standardize to Yes/No")
print("=" * 60)

CLASS 1 — PROFILING SUMMARY
Dataset size         : 5,001 customers, 21 attributes
Full row duplicates  : 1
ID duplicates        : 1
Columns with missing : 3
Overall churn rate   : 23.8%

📌 Confirmed issues for Class 2 (Cleaning):
  1. total_charges stored as object — needs type conversion
  2. Investigate structured missingness in total_charges
  3. senior_citizen stored as int — standardize to Yes/No


In [9]:
""" Dtype audit """
dtype_df = profile["dtype_audit"]
dtype_df

,column,stored_dtype,n_unique,sample_value,dtype_suspicious
0,customer_id,object,5000,CUST-00005,False
1,gender,object,5,Male,False
2,senior_citizen,int64,2,0,False
3,partner,object,2,Yes,False
4,dependents,object,2,Yes,False
5,tenure_months,int64,74,53,False
6,phone_service,object,2,Yes,False
7,multiple_lines,object,3,No,False
8,internet_service,object,3,No,False
9,online_security,object,3,No internet service,False


In [10]:
suspicious = dtype_df[dtype_df["dtype_suspicious"] == True]
if not suspicious.empty:
    print(f"\n⚠  {len(suspicious)} column(s) flagged as dtype-suspicious:")
    print(suspicious["column"].tolist())
else:
    print("\n✅ No dtype issues detected.")


⚠  1 column(s) flagged as dtype-suspicious:
['total_charges']


In [11]:
""" Missing value report """
missing_df = profile["missing_report"]
display(missing_df)
has_missing = missing_df[missing_df["combined_missing"] > 0]
if has_missing.empty:
    print("\n✅ No missing values detected.")
else:
    print(f"\n⚠  {len(has_missing)} column(s) have missing or blank values.")

,column,nan_count,blank_string_count,combined_missing,missing_pct
0,dependents,200,0,200,4.00
1,payment_method,150,0,150,3.00
2,total_charges,0,64,64,1.28
3,customer_id,0,0,0,0.00
4,device_protection,0,0,0,0.00
5,monthly_charges,0,0,0,0.00
6,paperless_billing,0,0,0,0.00
7,contract_type,0,0,0,0.00
8,streaming_movies,0,0,0,0.00
9,streaming_tv,0,0,0,0.00



⚠  3 column(s) have missing or blank values.


In [12]:
"""Duplicate report"""
dupe_report = profile["duplicate_report"]
print(f"Full row duplicates : {dupe_report['full_row_duplicates']}")
print(f"ID duplicates       : {dupe_report['id_duplicates']} (column: {dupe_report['id_column']})")

if dupe_report["id_duplicates"] > 0:
    print("\n⚠  Sample duplicated ID rows:")
    display(dupe_report["id_duplicate_examples"])
else:
    print("✅ No ID duplicates found.")

Full row duplicates : 1
ID duplicates       : 1 (column: customer_id)

⚠  Sample duplicated ID rows:


,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,CUST-00005,Male,0,Yes,Yes,53,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,29.29,1553.16,No
5,CUST-00005,Male,0,Yes,Yes,53,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,29.29,1553.16,No


In [13]:
"""Cardinality Report"""
display(profile["cardinality_report"])

,column,n_unique,pct_unique,inferred_type
0,customer_id,5000,99.98,high-cardinality
1,total_charges,4913,98.24,high-cardinality
2,monthly_charges,3414,68.27,high-cardinality
3,tenure_months,74,1.48,high-cardinality
4,gender,5,0.10,low-cardinality categorical
5,payment_method,4,0.08,low-cardinality categorical
6,contract_type,4,0.08,low-cardinality categorical
7,device_protection,3,0.06,low-cardinality categorical
8,streaming_movies,3,0.06,low-cardinality categorical
9,streaming_tv,3,0.06,low-cardinality categorical


In [14]:
"""Target distribution"""
display(profile["target_distribution"])

,class,count,percentage,imbalance_flag
0,No,3813,76.24,
1,Yes,1188,23.76,⚠ minority class


In [15]:
""" Generate reports/data_profile.md """
def generate_profile_report(profile: dict, output_path: str) -> None:
    """Write the full profile results to a markdown file."""
    shape = profile["shape"]
    missing_df = profile["missing_report"]
    dupe = profile["duplicate_report"]
    target_df = profile["target_distribution"]
    dtype_df = profile["dtype_audit"]
    suspicious = dtype_df[dtype_df["dtype_suspicious"] == True]

    lines = [
        "# Data Profile Report — Telecom Customer Churn Dataset\n",
        f"**Rows:** {shape['n_rows']:,} | **Columns:** {shape['n_columns']}\n",
        "\n---\n",
        "## 1. Dtype Audit\n",
    ]

    if suspicious.empty:
        lines.append("No dtype issues detected. All columns appear correctly typed.\n")
    else:
        lines.append(f"**{len(suspicious)} column(s) flagged as dtype-suspicious:**\n")
        for col in suspicious["column"].tolist():
            lines.append(f"- `{col}`\n")

    lines += ["\n## 2. Missing Value Summary\n"]
    has_missing = missing_df[missing_df["combined_missing"] > 0]
    if has_missing.empty:
        lines.append("No missing values detected across any column.\n")
    else:
        lines.append(has_missing.to_markdown(index=False) + "\n")

    lines += [
        "\n## 3. Duplicate Summary\n",
        f"- Full row duplicates: **{dupe['full_row_duplicates']}**\n",
        f"- ID-level duplicates (`{dupe['id_column']}`): **{dupe['id_duplicates']}**\n",
    ]

    lines += [
        "\n## 4. Target Variable Distribution\n",
        target_df.to_markdown(index=False) + "\n",
    ]

    lines += [
        "\n---\n",
        "*Generated programmatically by `src/profiler.py`. "
        "Do not edit manually — rerun the notebook to refresh.*\n"
    ]

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.writelines(lines)
    print(f"✅ Data profile report saved to {output_path}")


generate_profile_report(profile, output_path="../reports/data_profile.md")


✅ Data profile report saved to ../reports/data_profile.md
